In [ ]:
import numpy as np
import collections
from deepchem.molnet.load_function.clintox_datasets import load_clintox
from deepchem.molnet.load_function.tox21_datasets import load_tox21
from deepchem.data import NumpyDataset
from deepchem.feat import RawFeaturizer
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.info")

In [ ]:
# pull data without splitting, set molecular representation to SMILES
clintox_dataset = load_clintox(splitter=None, featurizer=RawFeaturizer(smiles=True))
tox21_dataset = load_tox21(splitter=None, featurizer=RawFeaturizer(smiles=True))

In [ ]:
print("First 20 rows of original ClinTox dataset:")

for i, (x, y, w, id) in enumerate(clintox_dataset[1][0].itersamples()):
    if i >= 20:
        break

    print("Row:", i)
    print("SMILES:", id)
    print("X:", x)
    print("y:", y)
    print("w:", w)

In [ ]:
_LARGEST_FRAGMENT = rdMolStandardize.LargestFragmentChooser()

def standardize_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    mol = _LARGEST_FRAGMENT.choose(mol)

    return Chem.MolToSmiles(mol)


def clean_dataset(toxicity_data, duplicate_mode):
    valid_indices = []
    standardized_smiles = []

    # maps standardized SMILES -> list of original dataset indices
    smiles_to_indices = {}

    for i, smiles in enumerate(toxicity_data.ids):
        standardized = standardize_smiles(smiles)

        if standardized is None:
            continue

        if standardized not in smiles_to_indices:
            smiles_to_indices[standardized] = []

        smiles_to_indices[standardized].append(i)

    for standardized, indices in smiles_to_indices.items():
        # keeping first duplicate instance for tox21
        if len(indices) == 1 or duplicate_mode == "keep_first":
            valid_indices.append(indices[0])
            standardized_smiles.append(standardized)

        # remove all duplicated for clintox
        elif duplicate_mode == "remove_all":
            continue
        
    toxicity_data_filtered = toxicity_data.select(valid_indices)

    toxicity_data_filtered = NumpyDataset(
        X=toxicity_data_filtered.X,
        y=toxicity_data_filtered.y,
        w=toxicity_data_filtered.w,
        ids=standardized_smiles
    )

    return toxicity_data_filtered

clintox_clean = clean_dataset(clintox_dataset[1][0], "remove_all")
tox21_clean = clean_dataset(tox21_dataset[1][0], "keep_first")

print(clintox_dataset[1][0].get_shape())
print(clintox_clean.get_shape())

print(tox21_dataset[1][0].get_shape())
print(tox21_clean.get_shape())
